# IPL LLM-Powered Structured Extraction and RAG

## Question 4 – LLM Structured Extraction and Retrieval-Augmented Generation

This project extends the IPL data work completed in the previous questions by implementing two LLM-powered pipelines:

1. LLM-powered structured information extraction with Pydantic validation.
2. Retrieval-Augmented Generation (RAG) using IPL-related documents.

The project demonstrates structured extraction, schema validation, error handling, document chunking, embeddings, vector search, retrieval, and grounded LLM-based question answering.

In [70]:
!pip install -q pydantic pandas numpy

# Prepare IPL Text Records

## Create 15+ IPL Text Records

### Step 2: IPL Text Records for Structured Extraction

In this step, we prepare more than 15 unstructured IPL match-related text records.

These text records will be passed to an LLM, which will extract important information into a predefined structured format.

In [71]:
ipl_text_records = [
    "Mumbai Indians defeated Chennai Super Kings by 5 wickets at Wankhede Stadium. Mumbai Indians successfully chased the target in the second innings.",

    "Chennai Super Kings won against Kolkata Knight Riders by 7 wickets at MA Chidambaram Stadium. Chennai Super Kings controlled the match from the beginning.",

    "Kolkata Knight Riders defeated Royal Challengers Bangalore by 6 wickets at Eden Gardens. Kolkata Knight Riders completed a successful run chase.",

    "Royal Challengers Bangalore defeated Rajasthan Royals by 8 runs at M. Chinnaswamy Stadium. The match was closely contested until the final over.",

    "Rajasthan Royals defeated Sunrisers Hyderabad by 5 wickets at Sawai Mansingh Stadium. Rajasthan Royals chased the target successfully.",

    "Delhi Capitals defeated Punjab Kings by 4 wickets at Arun Jaitley Stadium. Delhi Capitals performed strongly during the second innings.",

    "Sunrisers Hyderabad defeated Mumbai Indians by 3 wickets at Rajiv Gandhi International Stadium. Sunrisers Hyderabad won after a competitive run chase.",

    "Gujarat Titans defeated Lucknow Super Giants by 7 runs at Narendra Modi Stadium. Gujarat Titans defended their total successfully.",

    "Lucknow Super Giants defeated Rajasthan Royals by 2 wickets at Ekana Cricket Stadium. The match went down to the final over.",

    "Punjab Kings defeated Kolkata Knight Riders by 5 runs at Punjab Cricket Association Stadium. Punjab Kings successfully defended their target.",

    "Mumbai Indians defeated Royal Challengers Bangalore by 6 wickets at Wankhede Stadium. Mumbai Indians dominated the chase and won comfortably.",

    "Chennai Super Kings defeated Delhi Capitals by 27 runs at MA Chidambaram Stadium. Chennai Super Kings posted a strong total and defended it successfully.",

    "Kolkata Knight Riders defeated Sunrisers Hyderabad by 8 wickets at Eden Gardens. Kolkata Knight Riders completed the chase with several overs remaining.",

    "Royal Challengers Bangalore defeated Punjab Kings by 4 wickets at M. Chinnaswamy Stadium. Royal Challengers Bangalore chased the target successfully.",

    "Rajasthan Royals defeated Gujarat Titans by 3 wickets at Narendra Modi Stadium. Rajasthan Royals won a close match after a competitive chase.",

    "Delhi Capitals defeated Lucknow Super Giants by 6 runs at Arun Jaitley Stadium. Delhi Capitals defended a narrow target in a close contest.",

    "Sunrisers Hyderabad defeated Chennai Super Kings by 5 wickets at Rajiv Gandhi International Stadium. Sunrisers Hyderabad performed well in both innings.",

    "Gujarat Titans defeated Mumbai Indians by 6 runs at Narendra Modi Stadium. Gujarat Titans defended their total successfully in a close match.",

    "Lucknow Super Giants defeated Punjab Kings by 21 runs at Ekana Cricket Stadium. Lucknow Super Giants produced a strong all-round performance."
]

print("Total IPL text records:", len(ipl_text_records))

Total IPL text records: 19


### Display the Records

In [72]:
for i, text in enumerate(ipl_text_records, start=1):
    print(f"Record {i}:")
    print(text)
    print("-" * 80)

Record 1:
Mumbai Indians defeated Chennai Super Kings by 5 wickets at Wankhede Stadium. Mumbai Indians successfully chased the target in the second innings.
--------------------------------------------------------------------------------
Record 2:
Chennai Super Kings won against Kolkata Knight Riders by 7 wickets at MA Chidambaram Stadium. Chennai Super Kings controlled the match from the beginning.
--------------------------------------------------------------------------------
Record 3:
Kolkata Knight Riders defeated Royal Challengers Bangalore by 6 wickets at Eden Gardens. Kolkata Knight Riders completed a successful run chase.
--------------------------------------------------------------------------------
Record 4:
Royal Challengers Bangalore defeated Rajasthan Royals by 8 runs at M. Chinnaswamy Stadium. The match was closely contested until the final over.
--------------------------------------------------------------------------------
Record 5:
Rajasthan Royals defeated Sunriser

# Define the Pydantic Structured Output Schema

## Step 3: Define Pydantic Structured Output Schema

Pydantic is used to define and validate the structure of the information extracted by the LLM.

The schema ensures that every extracted IPL match record contains the required fields and follows the expected data types.

In [ ]:
from pydantic import BaseModel, ConfigDict, Field

class IPLMatchExtraction(BaseModel):
    model_config = ConfigDict(extra="forbid")
    teams: list[str] = Field(min_length=2, max_length=2, description="The two IPL teams that played")
    venue: str
    winner: str
    result: str
    summary: str

In [74]:
sample_record = IPLMatchExtraction(
    team="Mumbai Indians",
    opponent="Chennai Super Kings",
    venue="Wankhede Stadium",
    result="Mumbai Indians won by 5 wickets",
    summary="Mumbai Indians successfully chased the target in the second innings."
)

print(sample_record)

team='Mumbai Indians' opponent='Chennai Super Kings' venue='Wankhede Stadium' result='Mumbai Indians won by 5 wickets' summary='Mumbai Indians successfully chased the target in the second innings.'


In [75]:
print(sample_record.model_dump())

{'team': 'Mumbai Indians', 'opponent': 'Chennai Super Kings', 'venue': 'Wankhede Stadium', 'result': 'Mumbai Indians won by 5 wickets', 'summary': 'Mumbai Indians successfully chased the target in the second innings.'}


# Set Up the LLM

## Step 4: LLM Setup for Structured Extraction

In this step, we connect a Large Language Model (LLM) to the IPL structured extraction pipeline.

The LLM will read each unstructured IPL match description and extract the required information according to the Pydantic schema.

In [76]:
!pip install -q google-genai

In [77]:
from google import genai

print("Google GenAI library imported successfully.")

Google GenAI library imported successfully.


In [1]:
from getpass import getpass

GEMINI_API_KEY = getpass("Enter your Gemini API key: ")

In [114]:
client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client initialized successfully.")

Gemini client initialized successfully.


In [81]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explain in one sentence what the IPL is."
)

print(response.text)

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}

In [82]:
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-

In [83]:
response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="Explain in one sentence what the IPL is."
)

print(response.text)

The Indian Premier League (IPL) is a professional men's Twenty20 cricket league in India, featuring franchise-based teams representing different cities in a high-profile annual tournament.


## Step 5: Create the Structured Extraction Prompt

In [ ]:
def create_extraction_prompt(match_text):
    return f"""Extract the IPL match facts from the text below.
Return one JSON object with exactly these fields:
- teams: array of the two teams
- venue: stadium name only; exclude surrounding prose
- winner: winning team
- result: concise result, such as 'Mumbai Indians won by 5 wickets'
- summary: one short factual sentence
Do not infer missing details. If a field is absent, use an empty string (or [] for teams).

Text: {match_text}"""

In [85]:
test_match = """
Mumbai Indians defeated Chennai Super Kings by 5 wickets at Wankhede Stadium.
Mumbai Indians successfully chased the target in the second innings.
"""

prompt = create_extraction_prompt(test_match)

print(prompt)


You are an IPL data extraction assistant.

Read the following IPL match description and extract the required information.

IPL Match Description:

Mumbai Indians defeated Chennai Super Kings by 5 wickets at Wankhede Stadium.
Mumbai Indians successfully chased the target in the second innings.


Extract the following fields:

1. teams: The two IPL teams that played the match.
2. venue: The stadium or venue where the match was played.
3. winner: The team that won the match.
4. result: The exact match result, such as "Mumbai Indians won by 5 wickets".
5. summary: Write a short one-line summary of the match.

Return ONLY a JSON object with these fields:
teams
venue
winner
result
summary

Do not add markdown, explanations, or extra fields.



In [87]:
response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=prompt
)

print(response.text)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3-flash\nPlease retry in 11.794925004s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3-flash'}, 'qu

## Step 6: Structured Output with Pydantic

In [88]:
from pydantic import BaseModel, Field, ConfigDict
from typing import List


class IPLMatchExtraction(BaseModel):
    # Reject any fields that are not defined in this schema
    model_config = ConfigDict(extra="forbid")

    teams: List[str] = Field(
        description="The two IPL teams that played the match"
    )
    venue: str = Field(
        description="The stadium or venue where the match was played"
    )
    winner: str = Field(
        description="The IPL team that won the match"
    )
    result: str = Field(
        description="The result of the match"
    )
    summary: str = Field(
        description="A short summary of the match"
    )

In [89]:
import json

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=prompt,
    config={
        "response_mime_type": "application/json"
    }
)

print(response.text)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3-flash\nPlease retry in 7.489338643s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3-flash'}, 'quo

In [90]:
structured_data = json.loads(response.text)

print(structured_data)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [91]:
validated_record = IPLMatchExtraction.model_validate(structured_data)

print(validated_record)

teams=['Mumbai Indians', 'Chennai Super Kings'] venue='Wankhede Stadium' winner='Mumbai Indians' result='Mumbai Indians won by 5 wickets' summary='Mumbai Indians defeated Chennai Super Kings by 5 wickets at Wankhede Stadium after a successful run chase.'


In [92]:
validated_dict = validated_record.model_dump()

print(validated_dict)

{'teams': ['Mumbai Indians', 'Chennai Super Kings'], 'venue': 'Wankhede Stadium', 'winner': 'Mumbai Indians', 'result': 'Mumbai Indians won by 5 wickets', 'summary': 'Mumbai Indians defeated Chennai Super Kings by 5 wickets at Wankhede Stadium after a successful run chase.'}


## Step 7: Prepare 15+ IPL Text Records

In this step, IPL match records are converted into short natural-language descriptions.
These records will later be passed through the Gemini LLM for structured extraction and Pydantic validation.

In [93]:
# Create 15 IPL match text records for LLM processing

ipl_text_records = [
    "Mumbai Indians defeated Chennai Super Kings by 5 wickets at Wankhede Stadium after a successful run chase.",

    "Chennai Super Kings defeated Royal Challengers Bangalore by 6 wickets at M. Chinnaswamy Stadium.",

    "Kolkata Knight Riders defeated Sunrisers Hyderabad by 8 wickets at Eden Gardens.",

    "Rajasthan Royals defeated Delhi Capitals by 4 wickets at Sawai Mansingh Stadium.",

    "Royal Challengers Bangalore defeated Mumbai Indians by 7 wickets at Wankhede Stadium.",

    "Sunrisers Hyderabad defeated Punjab Kings by 5 runs at Rajiv Gandhi International Stadium.",

    "Gujarat Titans defeated Lucknow Super Giants by 7 wickets at Narendra Modi Stadium.",

    "Delhi Capitals defeated Kolkata Knight Riders by 4 wickets at Arun Jaitley Stadium.",

    "Punjab Kings defeated Rajasthan Royals by 5 wickets at Maharaja Yadavindra Singh International Cricket Stadium.",

    "Lucknow Super Giants defeated Mumbai Indians by 5 runs at Ekana Cricket Stadium.",

    "Chennai Super Kings defeated Gujarat Titans by 63 runs at MA Chidambaram Stadium.",

    "Royal Challengers Bangalore defeated Punjab Kings by 4 wickets at M. Chinnaswamy Stadium.",

    "Mumbai Indians defeated Rajasthan Royals by 6 wickets at Wankhede Stadium.",

    "Kolkata Knight Riders defeated Delhi Capitals by 7 wickets at Eden Gardens.",

    "Sunrisers Hyderabad defeated Gujarat Titans by 5 wickets at Rajiv Gandhi International Stadium."
]

print("Total IPL records:", len(ipl_text_records))

Total IPL records: 15


In [ ]:
for i, record in enumerate(ipl_text_records, start=1):
    print(f"Record {i}: {record}")

## Step 8: Gemini Structured Extraction

Every saved record below is a successful Gemini response parsed as JSON and validated against the Pydantic schema. Failed calls remain in the run report and are not written to structured_results.json.

In [ ]:
import json
import re
import time

MODEL = "gemini-2.5-flash-lite"

def call_with_retry(prompt, tries=5, json_mode=True):
    for attempt in range(tries):
        try:
            request = {"model": MODEL, "contents": prompt}
            if json_mode:
                request["config"] = {"response_mime_type": "application/json"}
            return client.models.generate_content(**request)
        except Exception as exc:
            message = str(exc)
            retry_after = re.search(r"retry in ([\d.]+)s", message, re.I)
            if "429" in message and attempt < tries - 1:
                delay = float(retry_after.group(1)) + 1 if retry_after else 2 ** attempt * 5
                time.sleep(delay)
            else:
                raise

def extract_and_validate(match_text):
    try:
        response = call_with_retry(create_extraction_prompt(match_text))
        validated = IPLMatchExtraction.model_validate(json.loads(response.text))
        return {"status":"success", "input_text":match_text,
                "validated_output":validated.model_dump(), "error":None}
    except Exception as exc:
        return {"status":"failed", "input_text":match_text,
                "validated_output":None, "error":str(exc)[:200]}

In [ ]:
test_result = extract_and_validate(ipl_text_records[0])

print(test_result)

In [ ]:
all_results = []
for i, record in enumerate(ipl_text_records, start=1):
    print(f"Processing record {i}/{len(ipl_text_records)}...")
    result = extract_and_validate(record)
    all_results.append(result)
    if result["status"] == "success":
        time.sleep(4)
    else:
        print("Failed:", result["error"])

success_count = sum(r["status"] == "success" for r in all_results)
failure_count = len(all_results) - success_count
print(f"{success_count} of {len(all_results)} succeeded; {failure_count} failed")

In [ ]:
success_count = sum(
    1 for result in all_results
    if result["status"] == "success"
)

failure_count = sum(
    1 for result in all_results
    if result["status"] == "failed"
)

print("Successful records:", success_count)
print("Failed records:", failure_count)

In [ ]:
# Display failed records and their errors

for i, result in enumerate(all_results, start=1):
    if result["status"] == "failed":
        print("=" * 80)
        print("Record:", i)
        print("Input:", result["input_text"])
        print("Error:", result["error"])

In [ ]:
test_result = extract_and_validate(ipl_text_records[0])
print(test_result)

In [ ]:
# Regex-based records were removed. Extraction results must come from Gemini and pass Pydantic validation.
structured_records = [r["validated_output"] for r in all_results if r["status"] == "success"]
validated_records = [IPLMatchExtraction.model_validate(r) for r in structured_records]
print("Gemini-validated records:", len(validated_records))

In [ ]:
# Validate every successful Gemini result again before saving.
validated_records = [IPLMatchExtraction.model_validate(r["validated_output"])
                     for r in all_results if r["status"] == "success"]
print("Validated Gemini records:", len(validated_records))

## Save 15 validate records

In [ ]:
import json

validated_data = [record.model_dump() for record in validated_records]
with open("structured_results.json", "w", encoding="utf-8") as f:
    json.dump(validated_data, f, indent=2, ensure_ascii=False)

print(f"Saved {len(validated_data)} successful Gemini records to structured_results.json")
print(f"Gemini success count: {success_count} of {len(all_results)}")

## Create the required validation failure

In [103]:
# Intentional malformed record for validation testing

bad_record = {
    "teams": ["Mumbai Indians", "Chennai Super Kings"],
    "venue": "Wankhede Stadium",
    "winner": "Mumbai Indians",
    "result": "Mumbai Indians won by 5 wickets",
    "summary": "Mumbai Indians defeated Chennai Super Kings.",
    "unexpected_field": "This field should not be allowed"
}

try:
    invalid_record = IPLMatchExtraction.model_validate(bad_record)
    print("Validation unexpectedly passed:", invalid_record)

except Exception as e:
    print("Validation failed as expected.")
    print("Error type:", type(e).__name__)
    print("Error details:")
    print(e)

Validation failed as expected.
Error type: ValidationError
Error details:
1 validation error for IPLMatchExtraction
unexpected_field
  Extra inputs are not permitted [type=extra_forbidden, input_value='This field should not be allowed', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden


# RAG Pipeline

In [104]:
!pip install -q langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter
print("LangChain text splitters imported successfully!")

LangChain text splitters imported successfully!


## Create documents

In [105]:
ipl_documents = [
"""
Mumbai Indians defeated Chennai Super Kings at Wankhede Stadium.
Rohit Sharma scored 78 runs and Jasprit Bumrah took 3 wickets.
Mumbai Indians won by 5 wickets.
""",

"""
Royal Challengers Bengaluru defeated Kolkata Knight Riders at Chinnaswamy Stadium.
Virat Kohli scored a match-winning fifty.
RCB won by 7 wickets.
""",

"""
Chennai Super Kings defeated Rajasthan Royals.
Ruturaj Gaikwad scored 82 runs.
CSK won by 25 runs.
""",

"""
Sunrisers Hyderabad defeated Delhi Capitals.
Travis Head scored a century.
SRH won comfortably.
""",

"""
Gujarat Titans defeated Lucknow Super Giants.
Shubman Gill played a captain's innings.
GT won by 6 wickets.
"""
]

print("Documents:", len(ipl_documents))

Documents: 5


## Create Chunking

In [106]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

chunks = []

for doc in ipl_documents:
    chunks.extend(text_splitter.split_text(doc))

print("Total chunks:", len(chunks))
print(chunks[:2])

Total chunks: 5
['Mumbai Indians defeated Chennai Super Kings at Wankhede Stadium.\nRohit Sharma scored 78 runs and Jasprit Bumrah took 3 wickets.\nMumbai Indians won by 5 wickets.', 'Royal Challengers Bengaluru defeated Kolkata Knight Riders at Chinnaswamy Stadium.\nVirat Kohli scored a match-winning fifty.\nRCB won by 7 wickets.']


## Create Embeddings

In [107]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(chunks)

print("Embedding shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (5, 384)


## Create FAISS Vector Store

In [108]:
import faiss
import numpy as np

embedding_matrix = np.array(embeddings).astype("float32")

index = faiss.IndexFlatL2(embedding_matrix.shape[1])
index.add(embedding_matrix)

print("Vectors stored:", index.ntotal)

Vectors stored: 5


## Retrieval Function

In [109]:
def retrieve(query, k=2):
    query_embedding = embedding_model.encode([query]).astype("float32")

    distances, indices = index.search(query_embedding, k)

    results = []
    for i in indices[0]:
        results.append(chunks[i])

    return results

## Test with 5 Queries

In [ ]:
queries = [
    "Who defeated Chennai Super Kings?",
    "Which player scored a century?",
    "Who won by 7 wickets?",
    "Which team won by 25 runs?",
    "Who captained Gujarat Titans?"
]

for q in queries:
    docs = retrieve(q, k=2)
    context = "\n---\n".join(docs)
    response = call_with_retry(
        ("Answer using ONLY the context below. If the context does not "
         "contain the answer, say 'I don't know.'\n\n"
         f"Context:\n{context}\n\nQuestion: {q}"),
        json_mode=False,
    )
    print("=" * 60)
    print("Question:", q)
    print("Top-1:", docs[0] if docs else "<none>")
    print("Top-2:", docs[1] if len(docs) > 1 else "<none>")
    print("Generated Answer:", response.text)

In [ ]:
import pandas as pd

# Expected answer clues are checked against the retrieved chunks, not generated answers.
expected_clues = [
    "Mumbai Indians", "Travis Head", "RCB won by 7 wickets",
    "CSK won by 25 runs", "Shubman Gill"
]
retrieval_assessment = []
for question, clue in zip(queries, expected_clues):
    hits = retrieve(question, k=2)
    retrieval_assessment.append({
        "Query": question,
        "Top-1 correct": bool(hits and clue.casefold() in hits[0].casefold()),
        "Top-2 correct": any(clue.casefold() in chunk.casefold() for chunk in hits[:2]),
    })
retrieval_assessment = pd.DataFrame(retrieval_assessment)
display(retrieval_assessment)
print("Top-1 correct:", int(retrieval_assessment["Top-1 correct"].sum()), "/", len(queries))
print("Top-2 correct:", int(retrieval_assessment["Top-2 correct"].sum()), "/", len(queries))

### Retrieval relevance from the archive's saved output

This table scores the retrieved chunks saved in the supplied notebook archive. The five-query retrieval was not rerun in this environment.

| Query | Right chunk in top-1 | Right chunk in top-2 |
|---|:---:|:---:|
| Who defeated Chennai Super Kings? | No | Yes |
| Which player scored a century? | No | Yes |
| Who won by 7 wickets? | No | No |
| Which team won by 25 runs? | No | Yes |
| Who captained Gujarat Titans? | Yes | Yes |
| **Total** | **1/5** | **4/5** |


# Current execution status

The supplied archive's `structured_results.json` is empty (`[]`), so it contains 0 Gemini-validated records. The revised Gemini extraction and answer-generation cells were not rerun during this update because no Gemini API key was available. The notebook prints the actual success count when run. The cached five-query retrieval snapshot scored top-1 1/5 and top-2 4/5; that retrieval was not rerun during this update.


# Conclusion

This notebook implements a complete workflow combining structured IPL information extraction with a Retrieval-Augmented Generation (RAG) pipeline.

The extraction code is configured to use Gemini and validates JSON results with Pydantic before saving them. The RAG code retrieves IPL text chunks and asks Gemini to answer using only that retrieved context. See the run-status note above for the available execution results.


In [ ]:
print("=" * 60)
print("PIPELINE STATUS")
print("=" * 60)
print(f"Gemini-validated records: {success_count} of {len(all_results)}")
print(f"RAG evaluation queries configured: {len(queries)}")
print("Generated answers are grounded in the retrieved context when the query cell runs.")
